# Track A, Part B, Data Handling (your turn)
Fill the `# TODO` lines, the **Your choice** cells, and the **Observe** cells. Cells without a prompt are demos to run. For every choice ask: would this be available when we forecast?

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.titleweight": "bold"})
BLUE, ORANGE, GREY = "#1f6feb", "#fb8500", "#9aa0a6"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"; IMG = ROOT / "images" / "partB"; IMG.mkdir(parents=True, exist_ok=True)
(ROOT / "models").mkdir(exist_ok=True)

## 1. Load and protect the raw data
Never overwrite the raw files. Load once, then work on copies.

In [ ]:
energy_raw = pd.read_csv(DATA / "energy_dataset.csv")
weather_raw = pd.read_csv(DATA / "weather_features.csv")
energy, weather = energy_raw.copy(), weather_raw.copy()
print("energy", energy.shape, "| weather", weather.shape)

## 2. Timeline integrity
Parse to UTC, sort, and check the hours are regular. "Same hour yesterday" only means something if yesterday's hour is there.

In [ ]:
energy["time"]  = pd.to_datetime(energy["time"], utc=True, errors="coerce")
weather["time"] = pd.to_datetime(weather["dt_iso"], utc=True, errors="coerce")
energy = energy.sort_values("time").reset_index(drop=True)
full = pd.date_range(energy["time"].min(), energy["time"].max(), freq="h", tz="UTC")
print("duplicate hours:", int(energy["time"].duplicated().sum()),
      "| missing hours:", len(full.difference(pd.DatetimeIndex(energy["time"]))))

**Observe.** Are there duplicate or missing hours? Why would a single missing hour break a t-24 lag?

## 3. Target, reference, horizon
**Target** `total load actual` (MW). **Reference** `total load forecast`, a forecast already in the data, kept for comparison only. **Horizon** 24 hours.

In [ ]:
TARGET, REFERENCE, HORIZON = "total load actual", "total load forecast", 24
p = energy[[TARGET, REFERENCE]].dropna()
mape = ((p[REFERENCE] - p[TARGET]).abs() / p[TARGET]).mean() * 100
print(f"reference forecast MAPE = {mape:.2f}%  |  target missing values: {int(energy[TARGET].isna().sum())}")

**Observe.** How strong is the provided forecast? Should we use it as a feature? Why not?

## 4. Drop dead columns
**Your turn:** find the fully empty and the always zero columns. Trap: an all-NaN column has an empty `dropna()`, and `(empty == 0).all()` returns `True`, so exclude the empty ones from the zero test.

In [ ]:
empty_cols = [c for c in energy.columns if ...]                      # TODO: fully empty (100% NaN)
num = energy.select_dtypes("number")
zero_cols = [c for c in num.columns if c not in empty_cols and ...]  # TODO: always zero (exclude empty!)
energy = energy.drop(columns=empty_cols + zero_cols)
print("dropped", len(empty_cols + zero_cols), "->", energy.shape[1], "cols")

## 5. Weather to one row per hour, then merge
National target, so we need one weather row per hour. **Your turn:** aggregate across cities (one row per hour), then merge on `time`.

In [ ]:
weather["city_name"] = weather["city_name"].str.strip()
weather = weather.drop_duplicates(subset=["time", "city_name"])
wnum = [c for c in weather.select_dtypes("number").columns if c != "weather_id"]
weather_agg = ...   # TODO: mean across cities, one row per hour (groupby time)
for c in ["temp", "temp_min", "temp_max"]: weather_agg[c] -= 273.15
df = ...            # TODO: merge energy + weather_agg on "time" (left join)
print("df", df.shape)

## 6. Look first: distributions and box plots
Before counting missing values, we look. A box plot shows the spread and what reads as extreme. A profile by hour shows whether an extreme value is just a normal peak.

In [ ]:
df["hour"] = df["time"].dt.hour
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.boxplot(column=TARGET, ax=ax[0]); ax[0].set_title("total load actual, overall"); ax[0].set_ylabel("MW")
df.boxplot(column=TARGET, by="hour", ax=ax[1]); ax[1].set_title("total load actual, by hour"); ax[1].set_xlabel("hour")
plt.suptitle(""); fig.tight_layout(); fig.savefig(IMG / "box_target.png", dpi=130); plt.show()

**Observe.** Are the high target values errors, or normal peaks? Look at the by-hour view before deciding.

## 7. Missing values: where, how much, isolated or clustered
Now we count, and we look at when the gaps happen, because in a time series timing matters as much as quantity.

In [ ]:
rate = df.isna().mean() * 100; rate = rate[rate > 0].sort_values(ascending=False)
print(rate.round(2).to_string())
miss_t = df.set_index("time").isna().sum(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
rate.head(10).iloc[::-1].plot.barh(ax=ax[0], color=ORANGE); ax[0].set_title("missing rate by column (%)")
ax[1].plot(miss_t.index, miss_t.values, color=GREY, lw=0.8); ax[1].set_title("missing values per timestamp")
fig.tight_layout(); fig.savefig(IMG / "missing.png", dpi=130); plt.show()

**Observe.** How many gaps, and are they isolated spikes or solid blocks? What does that imply for how we fill them?

## 8. Choosing an imputation method
 Each method fits a situation.

| Method | Good when | Watch out |
|---|---|---|
| median or mean | numeric feature, order does not matter | the mean is pulled by outliers |
| forward fill (ffill) | short gap in a continuous signal | repeats the last past value, fine |
| time interpolation | short isolated gaps, smooth signal | uses neighbours, a mild assumption |
| back fill (bfill) | never in forecasting | uses the future, forbidden |
| drop the column | the column is almost all empty | loses signal |
| flag and exclude | the target, which is a label | shrinks the labelled set |

Run the example, then make your choice below.

In [ ]:
prof = df.groupby("hour")["generation solar"].mean()
gmed = df["generation solar"].median()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(prof.index, prof.values, marker="o", color=BLUE, label="real hourly profile")
ax.axhline(gmed, color=ORANGE, ls="--", label=f"global median = {gmed:.0f}")
ax.set_title("generation solar by hour: the median ignores the cycle")
ax.set_xlabel("hour"); ax.set_ylabel("MW"); ax.legend()
fig.savefig(IMG / "impute_choice.png", dpi=130); plt.show()

**Your choice.**
Generation features (isolated gaps, strong daily cycle): method = ____ , why = ____
Production pipeline (scores one row at a time): method = ____ , why = ____

## 9. The target is special
The target is a label. Naively imputing it fabricates demand that never happened. Dropping its 36 rows breaks the hourly continuity, so lags would misalign.
**Your turn:** propose a strategy that keeps the series regular for features AND never lets an invented value be trained on or scored. Then fill the `# TODO`.

In [ ]:
df["target_missing"] = df[TARGET].isna()
df["target_filled"]  = ...   # TODO: keep the series regular without inventing scored values (hint: interpolate over time)
print("target gaps:", int(df["target_missing"].sum()))

## 10. Treat outliers
These bad values are present, not missing, so the imputer never touches them (it only fills NaN). We check the weather variables are physically plausible, and clip the impossible ones.

In [ ]:
pc, ph = weather["pressure"].max(), df["pressure"].max()
wc, wh = weather["wind_speed"].max(), df["wind_speed"].max()
print(f"pressure: worst single city {pc:>10,.0f} hPa  ->  hourly mean still {ph:>9,.0f}   (real near 1013, the error PROPAGATED through the mean)")
print(f"wind    : worst single city {wc:>10.0f} m/s  ->  hourly mean only  {wh:>9.1f}   (storm near 30, the error was HIDDEN by the mean)")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
df.boxplot(column="pressure", ax=ax[0]); ax[0].set_title("pressure BEFORE clip (one impossible value blows up the scale)")
df["pressure"]   = df["pressure"].clip(870, 1085)   # fixed physical bounds, not learned, so no leakage
df["wind_speed"] = df["wind_speed"].clip(0, 40)
df.boxplot(column="pressure", ax=ax[1]); ax[1].set_title("pressure AFTER clip (plausible range near 1013)")
plt.suptitle(""); fig.tight_layout(); fig.savefig(IMG / "outliers.png", dpi=130); plt.show()

**Observe.** Which bad value survived the averaging, and which one was hidden? Why can the imputer not fix these?

## 11. Split by time, never random
Shuffling lets the model peek at neighbouring hours, which inflates every score. **Your turn:** train up to 2016, validation 2017, test 2018.

In [ ]:
year = df["time"].dt.year
train_df = ...   # TODO: year <= 2016
val_df   = ...   # TODO: year == 2017
test_df  = ...   # TODO: year == 2018
for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{n:5s} {d.shape[0]:6d}   {d['time'].min().date()} -> {d['time'].max().date()}")

**Observe.** What are the three sizes, and why must the test year stay untouched?

## 12. Choosing a scaler
Scaling puts features on a comparable footing. Not every model needs it.

| Scaler | What it does | Good when |
|---|---|---|
| StandardScaler | mean 0, std 1 | linear / Ridge / neural nets, the usual default |
| MinMaxScaler | rescales to [0, 1] | bounded inputs, but one outlier squashes the rest |
| RobustScaler | centers on median, scales by the IQR | features still skewed or with outliers |
| none | leave as is | tree models (RF, GBM, XGBoost) do not need scaling |

Linear and gradient based models need scaling; tree based models do not, and scaling does not hurt them.

In [ ]:
print(df[[TARGET, "temp", "price actual"]].describe().loc[["min", "max"]].round(1).to_string())
print("\nThe features live on very different scales (load in tens of thousands, temp in tens), so a linear model needs scaling.")

**Your choice.**
Scaler for a Ridge baseline = ____ , why = ____
Do the tree models (RF, GBM) need scaling? ____ , why = ____

## 13. Fit transforms on train only
Anything that learns from data must learn on train only, then apply to validation and test. **Your turn:** fit the imputer and the chosen scaler on train, transform validation and test.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
feat = [c for c in df.select_dtypes("number").columns if c not in {TARGET, REFERENCE, "target_filled"}]
imp, sc = SimpleImputer(strategy="median"), StandardScaler()
X_train = ...   # TODO: fit_transform on train (impute then scale)
X_val   = ...   # TODO: transform val
X_test  = ...   # TODO: transform test
print("solar median  train:", round(train_df['generation solar'].median(), 1),
      "| all-data:", round(df['generation solar'].median(), 1), " <- fitting on all would leak val and test")

**Observe.** Are the two medians the same? What does the difference tell you about fitting on all the data?

## 14. Wrap it in a pipeline and save it
One fitted object gives the same transform for train, validation, test, and the FastAPI backend. We save it so the next notebook reloads it instead of retraining.

In [ ]:
from sklearn.pipeline import Pipeline
import joblib
preprocess = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
preprocess.fit(train_df[feat])
joblib.dump({"pipeline": preprocess, "features": feat}, ROOT / "models" / "preprocess.joblib")
print("pipeline:", [n for n, _ in preprocess.steps], "| saved to models/preprocess.joblib")

## 15. Leakage and availability audit
For each family, answer: available at prediction time? then give a use. This is the most important table of the day.

| Column or family | Available at prediction time? | Use |
|---|---|---|
| `total load actual` (target) |  |  |
| `total load forecast` |  |  |
| `generation *` (actual) |  |  |
| `forecast * day ahead` |  |  |
| `price day ahead` |  |  |
| `price actual` |  |  |
| weather (observed) |  |  |
| calendar (hour, dow, month) |  |  |

**Observe.** Which families are safe to feed the model directly, and which are off limits because they carry the future?

## 16. Done, next is feature engineering
Protocol built: clean structure, weather merged, visuals first, missing values and the scaler chosen with justification, the target handled without breaking continuity, outliers clipped, time split, train-only transforms, pipeline saved, leakage audited.

Next session: calendar features, lags (t-24, t-168), rolling statistics (past only), the 24 hour target, baselines, and the first models.